# Time Series com MongoDB

Este notebook é um guia prático de **Time Series Collections** no MongoDB, introduzidas na versão 5.0. O conteúdo é organizado em níveis crescentes de complexidade.

## O que você vai aprender

| Seção | Conteúdo |
|---|---|
| 1 | Setup: iniciando o MongoDB e conectando com PyMongo |
| 2 | Operações básicas: criar coleção, inserir e consultar documentos |
| 3 | Filtros e agregações intermediárias |
| 4 | Janelas temporais e estatísticas avançadas |
| 5 | Caso real: pipeline completo para dados de sensores IoT |

## Por que Time Series Collections?

Coleções comuns do MongoDB armazenam cada leitura como um documento separado, o que gera sobrecarga de espaço e índice para séries temporais de alta frequência. As **Time Series Collections** resolvem isso internamente agrupando múltiplas medições no mesmo "bucket" de armazenamento, resultando em:

- **Menor uso de disco** (compressão automática por coluna)
- **Queries mais rápidas** em ranges de tempo
- **API idêntica** ao MongoDB regular — sem curva de aprendizado nova

> **Pré-requisito:** MongoDB 5.0+. Este lab usa MongoDB 8.3 instalado via `postBuild`.

---
## Seção 1 — Setup

### 1.1 Instalar dependências

In [ ]:
!pip install pymongo --quiet

### 1.2 Iniciar o servidor MongoDB

O MongoDB foi instalado em `~/resources/local/` pelo script `postBuild`. Execute a célula abaixo para iniciá-lo em segundo plano.

In [ ]:
import os
import subprocess
import time

mongodb_home = os.environ.get('MONGODB_HOME', os.path.expanduser('~/resources/local/mongodb-8.3.1'))
data_dir     = os.path.expanduser('~/resources/local/mongodb-data')
log_file     = os.path.expanduser('~/resources/local/mongodb.log')

os.makedirs(data_dir, exist_ok=True)

mongod_bin = os.path.join(mongodb_home, 'bin', 'mongod')

proc = subprocess.Popen(
    [mongod_bin, '--dbpath', data_dir, '--logpath', log_file, '--fork'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
stdout, stderr = proc.communicate()
print(stdout.decode())

time.sleep(2)
print('MongoDB iniciado. Log em:', log_file)

### 1.3 Conectar ao MongoDB com PyMongo

In [ ]:
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')

# Verifica a conexão
info = client.server_info()
print(f"Conectado ao MongoDB versão {info['version']}")

---
## Seção 2 — Operações Básicas

### 2.1 Criar um banco de dados e uma Time Series Collection

Uma Time Series Collection exige três parâmetros obrigatórios:

| Parâmetro | Descrição |
|---|---|
| `timeField` | Campo com o timestamp de cada medição (deve ser `datetime`) |
| `metaField` | Campo que identifica a série (ex: id do sensor) |
| `granularity` | Granularidade dos dados: `"seconds"`, `"minutes"` ou `"hours"` |

In [ ]:
db = client['timeseries_lab']

# Remove a coleção se já existir (facilita re-execução do notebook)
if 'temperatura' in db.list_collection_names():
    db.drop_collection('temperatura')

db.create_collection(
    'temperatura',
    timeseries={
        'timeField':   'timestamp',
        'metaField':   'sensor_id',
        'granularity': 'seconds'
    }
)

colecao = db['temperatura']
print('Coleção "temperatura" criada com sucesso.')

### 2.2 Inserir um único documento

Cada documento deve conter o campo `timeField` como um objeto `datetime` com timezone UTC.

In [ ]:
from datetime import datetime, timezone

documento = {
    'timestamp': datetime(2024, 1, 15, 10, 0, 0, tzinfo=timezone.utc),
    'sensor_id': 'sensor-A1',
    'temperatura_c': 22.5,
    'umidade_pct':   65.0,
    'localizacao': 'sala-servidores'
}

resultado = colecao.insert_one(documento)
print('Documento inserido, id:', resultado.inserted_id)

### 2.3 Inserir múltiplos documentos de uma vez

`insert_many` é sempre preferível a inserções em loop para melhor desempenho.

In [ ]:
from datetime import timedelta
import random

random.seed(42)

sensores    = ['sensor-A1', 'sensor-A2', 'sensor-B1']
inicio      = datetime(2024, 1, 15, 8, 0, 0, tzinfo=timezone.utc)
documentos  = []

# Gera 3 horas de leituras a cada 1 minuto para cada sensor
for minuto in range(180):
    ts = inicio + timedelta(minutes=minuto)
    for sensor in sensores:
        documentos.append({
            'timestamp':     ts,
            'sensor_id':     sensor,
            'temperatura_c': round(20.0 + random.uniform(-2, 5), 2),
            'umidade_pct':   round(60.0 + random.uniform(-5, 10), 2),
            'localizacao':   'sala-servidores' if sensor.startswith('sensor-A') else 'datacenter'
        })

colecao.insert_many(documentos)
print(f'{len(documentos)} documentos inseridos.')

### 2.4 Consultas simples com `find`

A API de consulta é idêntica a qualquer outra coleção MongoDB.

In [ ]:
# Busca as 5 leituras mais recentes do sensor-A1
cursor = (
    colecao
    .find({'sensor_id': 'sensor-A1'}, {'_id': 0, 'timestamp': 1, 'temperatura_c': 1, 'umidade_pct': 1})
    .sort('timestamp', -1)
    .limit(5)
)

for doc in cursor:
    print(doc)

In [ ]:
# Conta o total de documentos por sensor
total = colecao.count_documents({})
print(f'Total de documentos na coleção: {total}')

for sensor in sensores:
    qtd = colecao.count_documents({'sensor_id': sensor})
    print(f'  {sensor}: {qtd} leituras')

---
## Seção 3 — Filtros e Agregações Intermediárias

### 3.1 Filtrar por intervalo de tempo

O operador `$gte` / `$lte` no campo `timeField` é a base de qualquer query em séries temporais. O MongoDB usa o índice de tempo automaticamente.

In [ ]:
inicio_janela = datetime(2024, 1, 15,  9, 0, 0, tzinfo=timezone.utc)
fim_janela    = datetime(2024, 1, 15, 10, 0, 0, tzinfo=timezone.utc)

cursor = colecao.find(
    {
        'timestamp': {'$gte': inicio_janela, '$lt': fim_janela},
        'sensor_id': 'sensor-A1'
    },
    {'_id': 0, 'timestamp': 1, 'temperatura_c': 1}
).sort('timestamp', 1)

docs = list(cursor)
print(f'{len(docs)} leituras entre 09:00 e 10:00 para sensor-A1')
for doc in docs[:5]:
    print(' ', doc)

### 3.2 Agregação: média por sensor

O estágio `$group` é a base do aggregation framework. Aqui calculamos temperatura e umidade médias por sensor.

In [ ]:
pipeline = [
    {
        '$group': {
            '_id':              '$sensor_id',
            'temp_media':       {'$avg': '$temperatura_c'},
            'temp_maxima':      {'$max': '$temperatura_c'},
            'temp_minima':      {'$min': '$temperatura_c'},
            'umidade_media':    {'$avg': '$umidade_pct'},
            'total_leituras':   {'$sum': 1}
        }
    },
    {'$sort': {'_id': 1}}
]

for resultado in colecao.aggregate(pipeline):
    print(f"Sensor: {resultado['_id']}")
    print(f"  Temperatura — min: {resultado['temp_minima']:.2f}°C  "
          f"média: {resultado['temp_media']:.2f}°C  "
          f"max: {resultado['temp_maxima']:.2f}°C")
    print(f"  Umidade média: {resultado['umidade_media']:.2f}%")
    print(f"  Total de leituras: {resultado['total_leituras']}")

### 3.3 Downsampling — Agrupar leituras por hora

Downsampling é a técnica de reduzir a resolução temporal dos dados, trocando detalhes por performance em consultas de longo prazo. Usamos `$dateTrunc` para truncar o timestamp à hora.

In [ ]:
pipeline = [
    {
        '$group': {
            '_id': {
                'hora':   {'$dateTrunc': {'date': '$timestamp', 'unit': 'hour'}},
                'sensor': '$sensor_id'
            },
            'temp_media':   {'$avg': '$temperatura_c'},
            'temp_max':     {'$max': '$temperatura_c'},
            'temp_min':     {'$min': '$temperatura_c'},
            'leituras':     {'$sum': 1}
        }
    },
    {'$sort': {'_id.hora': 1, '_id.sensor': 1}}
]

import pandas as pd

rows = []
for r in colecao.aggregate(pipeline):
    rows.append({
        'hora':       r['_id']['hora'].strftime('%H:%M'),
        'sensor':     r['_id']['sensor'],
        'temp_media': round(r['temp_media'], 2),
        'temp_max':   round(r['temp_max'],   2),
        'temp_min':   round(r['temp_min'],   2),
        'leituras':   r['leituras']
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

### 3.4 Filtrar leituras acima de um limiar (alertas)

Combinando `$match` com condições compostas para identificar eventos de alerta.

In [ ]:
LIMIAR_TEMP    = 24.0
LIMIAR_UMIDADE = 68.0

pipeline = [
    {
        '$match': {
            '$or': [
                {'temperatura_c': {'$gt': LIMIAR_TEMP}},
                {'umidade_pct':   {'$gt': LIMIAR_UMIDADE}}
            ]
        }
    },
    {
        '$addFields': {
            'alerta_temp':    {'$gt': ['$temperatura_c', LIMIAR_TEMP]},
            'alerta_umidade': {'$gt': ['$umidade_pct',   LIMIAR_UMIDADE]}
        }
    },
    {'$sort':  {'timestamp': 1}},
    {'$limit': 8},
    {'$project': {'_id': 0, 'timestamp': 1, 'sensor_id': 1,
                  'temperatura_c': 1, 'umidade_pct': 1,
                  'alerta_temp': 1, 'alerta_umidade': 1}}
]

for doc in colecao.aggregate(pipeline):
    alertas = []
    if doc['alerta_temp']:    alertas.append('TEMP_ALTA')
    if doc['alerta_umidade']: alertas.append('UMIDADE_ALTA')
    print(f"{doc['timestamp'].strftime('%H:%M')} | {doc['sensor_id']} | "
          f"T={doc['temperatura_c']}°C | U={doc['umidade_pct']}% | {', '.join(alertas)}")

---
## Seção 4 — Janelas Temporais e Estatísticas Avançadas

O operador `$setWindowFields` (MongoDB 5.0+) permite calcular estatísticas dentro de janelas deslizantes sem precisar de múltiplas queries. É o equivalente às **window functions** do SQL.

### 4.1 Média móvel (Moving Average)

A média móvel suaviza ruídos em séries temporais. Aqui calculamos a média das últimas 5 leituras para cada sensor.

In [ ]:
pipeline = [
    {'$match':  {'sensor_id': 'sensor-A1'}},
    {'$sort':   {'timestamp': 1}},
    {
        '$setWindowFields': {
            'partitionBy': '$sensor_id',
            'sortBy':      {'timestamp': 1},
            'output': {
                'media_movel_5': {
                    '$avg': '$temperatura_c',
                    'window': {'documents': [-4, 0]}  # janela: documento atual + 4 anteriores
                }
            }
        }
    },
    {'$limit': 10},
    {'$project': {'_id': 0, 'timestamp': 1, 'temperatura_c': 1, 'media_movel_5': 1}}
]

print(f"{'Horário':<10}  {'Temp (°C)':<12}  {'Média Móvel 5pts (°C)':<22}")
print('-' * 47)
for doc in colecao.aggregate(pipeline):
    print(f"{doc['timestamp'].strftime('%H:%M'):<10}  "
          f"{doc['temperatura_c']:<12.2f}  "
          f"{doc['media_movel_5']:<22.2f}")

### 4.2 Soma acumulada (Cumulative Sum)

Útil para acumular eventos ao longo do tempo, como totais de consumo.

In [ ]:
pipeline = [
    {'$match':  {'sensor_id': 'sensor-A1'}},
    {'$sort':   {'timestamp': 1}},
    {
        '$setWindowFields': {
            'partitionBy': '$sensor_id',
            'sortBy':      {'timestamp': 1},
            'output': {
                'soma_acumulada_temp': {
                    '$sum': '$temperatura_c',
                    'window': {'documents': ['unbounded', 'current']}  # do início até o atual
                },
                'numero_leitura': {
                    '$sum': 1,
                    'window': {'documents': ['unbounded', 'current']}
                }
            }
        }
    },
    {'$limit': 8},
    {'$project': {'_id': 0, 'timestamp': 1, 'temperatura_c': 1,
                  'soma_acumulada_temp': 1, 'numero_leitura': 1}}
]

print(f"{'Horário':<10}  {'Temp':<8}  {'Soma Acum.':<14}  {'Nº Leitura':<12}")
print('-' * 48)
for doc in colecao.aggregate(pipeline):
    print(f"{doc['timestamp'].strftime('%H:%M'):<10}  "
          f"{doc['temperatura_c']:<8.2f}  "
          f"{doc['soma_acumulada_temp']:<14.2f}  "
          f"{doc['numero_leitura']:<12}")

### 4.3 Desvio da leitura anterior (Delta)

Detecta variações bruscas comparando cada leitura com a anterior usando `$shift`.

In [ ]:
pipeline = [
    {'$match':  {'sensor_id': 'sensor-A1'}},
    {'$sort':   {'timestamp': 1}},
    {
        '$setWindowFields': {
            'partitionBy': '$sensor_id',
            'sortBy':      {'timestamp': 1},
            'output': {
                'leitura_anterior': {
                    '$shift': {
                        'output': '$temperatura_c',
                        'by':     -1,          # documento anterior
                        'default': None
                    }
                }
            }
        }
    },
    {
        '$addFields': {
            'delta_temp': {
                '$cond': {
                    'if':   {'$ne': ['$leitura_anterior', None]},
                    'then': {'$subtract': ['$temperatura_c', '$leitura_anterior']},
                    'else': None
                }
            }
        }
    },
    # Alerta: variação maior que 1°C em 1 minuto
    {'$match': {'$expr': {'$gt': [{'$abs': '$delta_temp'}, 1.0]}}},
    {'$limit': 8},
    {'$project': {'_id': 0, 'timestamp': 1, 'temperatura_c': 1,
                  'leitura_anterior': 1, 'delta_temp': 1}}
]

print(f"{'Horário':<10}  {'Atual':<8}  {'Anterior':<10}  {'Delta':<8}")
print('-' * 42)
for doc in colecao.aggregate(pipeline):
    print(f"{doc['timestamp'].strftime('%H:%M'):<10}  "
          f"{doc['temperatura_c']:<8.2f}  "
          f"{doc['leitura_anterior']:<10.2f}  "
          f"{doc['delta_temp']:+.2f}°C  <-- variação brusca")

### 4.4 Percentis e distribuição estatística

Calcula percentis para entender a distribuição das temperaturas — útil para definir limiares de alerta baseados em dados reais.

In [ ]:
pipeline = [
    {
        '$group': {
            '_id':    '$sensor_id',
            'p50':    {'$percentile': {'input': '$temperatura_c', 'p': [0.50], 'method': 'approximate'}},
            'p90':    {'$percentile': {'input': '$temperatura_c', 'p': [0.90], 'method': 'approximate'}},
            'p99':    {'$percentile': {'input': '$temperatura_c', 'p': [0.99], 'method': 'approximate'}},
            'desvio': {'$stdDevPop': '$temperatura_c'},
            'media':  {'$avg': '$temperatura_c'}
        }
    },
    {'$sort': {'_id': 1}}
]

print(f"{'Sensor':<12}  {'Média':>7}  {'p50':>7}  {'p90':>7}  {'p99':>7}  {'StdDev':>8}")
print('-' * 58)
for r in colecao.aggregate(pipeline):
    print(f"{r['_id']:<12}  "
          f"{r['media']:.2f}°C  "
          f"{r['p50'][0]:.2f}°C  "
          f"{r['p90'][0]:.2f}°C  "
          f"{r['p99'][0]:.2f}°C  "
          f"{r['desvio']:.4f}")

---
## Seção 5 — Caso Real: Pipeline Completo para Sensores IoT

Nesta seção simulamos um cenário real: monitoramento de temperatura em múltiplas salas de um datacenter, com dados vindos de vários sensores. O objetivo é gerar um **relatório de turno** (8h às 11h) com downsampling, alertas e ranking.

### 5.1 Criar coleção de sensores IoT com dados realistas

In [ ]:
import math

if 'sensores_iot' in db.list_collection_names():
    db.drop_collection('sensores_iot')

db.create_collection(
    'sensores_iot',
    timeseries={
        'timeField':   'ts',
        'metaField':   'meta',
        'granularity': 'seconds'
    }
)

iot = db['sensores_iot']

config_sensores = [
    {'id': 'rack-A-top',    'sala': 'sala-1', 'base_temp': 28.0, 'amplitude': 3.0},
    {'id': 'rack-A-bottom', 'sala': 'sala-1', 'base_temp': 24.0, 'amplitude': 2.0},
    {'id': 'rack-B-top',    'sala': 'sala-2', 'base_temp': 30.0, 'amplitude': 4.0},
    {'id': 'rack-B-bottom', 'sala': 'sala-2', 'base_temp': 25.0, 'amplitude': 2.5},
    {'id': 'corredor-frio', 'sala': 'sala-1', 'base_temp': 18.0, 'amplitude': 1.0},
]

random.seed(7)
inicio = datetime(2024, 1, 15, 8, 0, 0, tzinfo=timezone.utc)
docs   = []

for seg in range(0, 3 * 3600, 30):  # leitura a cada 30 segundos por 3 horas
    ts = inicio + timedelta(seconds=seg)
    for cfg in config_sensores:
        # Simula tendência de aquecimento ao longo do turno + ruído
        tendencia = (seg / 3600) * 1.5
        ruido     = random.gauss(0, 0.3)
        onda      = cfg['amplitude'] * math.sin(2 * math.pi * seg / 3600)
        temp      = round(cfg['base_temp'] + tendencia + onda + ruido, 2)

        docs.append({
            'ts':   ts,
            'meta': {'sensor_id': cfg['id'], 'sala': cfg['sala']},
            'temperatura_c': temp,
            'cpu_load_pct':  round(random.uniform(20, 95), 1)
        })

iot.insert_many(docs)
print(f'{len(docs)} leituras IoT inseridas para {len(config_sensores)} sensores.')

### 5.2 Relatório de turno: agregação por sala e período de 15 minutos

In [ ]:
pipeline = [
    # Apenas o turno da manhã
    {
        '$match': {
            'ts': {
                '$gte': datetime(2024, 1, 15,  8, 0, tzinfo=timezone.utc),
                '$lt':  datetime(2024, 1, 15, 11, 0, tzinfo=timezone.utc)
            }
        }
    },
    # Agrupa por sala + janela de 15 minutos
    {
        '$group': {
            '_id': {
                'sala':    '$meta.sala',
                'periodo': {
                    '$dateTrunc': {'date': '$ts', 'unit': 'minute', 'binSize': 15}
                }
            },
            'temp_media': {'$avg': '$temperatura_c'},
            'temp_max':   {'$max': '$temperatura_c'},
            'leituras':   {'$sum': 1}
        }
    },
    {'$sort': {'_id.periodo': 1, '_id.sala': 1}},
    # Flag de alerta se temperatura média acima de 29°C
    {
        '$addFields': {
            'em_alerta': {'$gt': ['$temp_max', 29.0]}
        }
    }
]

rows = []
for r in iot.aggregate(pipeline):
    rows.append({
        'periodo':    r['_id']['periodo'].strftime('%H:%M'),
        'sala':       r['_id']['sala'],
        'temp_media': round(r['temp_media'], 2),
        'temp_max':   round(r['temp_max'],   2),
        'leituras':   r['leituras'],
        'alerta':     '*** ALERTA ***' if r['em_alerta'] else ''
    })

df_relatorio = pd.DataFrame(rows)
print(df_relatorio.to_string(index=False))

### 5.3 Ranking de sensores mais críticos do turno

In [ ]:
pipeline = [
    {
        '$group': {
            '_id':        '$meta.sensor_id',
            'sala':       {'$first': '$meta.sala'},
            'temp_media': {'$avg':   '$temperatura_c'},
            'temp_max':   {'$max':   '$temperatura_c'},
            'temp_min':   {'$min':   '$temperatura_c'},
            'qtd_alertas': {
                '$sum': {'$cond': [{'$gt': ['$temperatura_c', 29.0]}, 1, 0]}
            }
        }
    },
    # Adiciona campo calculado: score de criticidade
    {
        '$addFields': {
            'score_criticidade': {
                '$add': [
                    '$temp_media',
                    {'$multiply': ['$qtd_alertas', 0.1]}
                ]
            }
        }
    },
    {'$sort':    {'score_criticidade': -1}},
    {'$project': {'_id': 0, 'sensor': '$_id', 'sala': 1,
                  'temp_media': 1, 'temp_max': 1, 'temp_min': 1,
                  'qtd_alertas': 1, 'score_criticidade': 1}}
]

print("Ranking de sensores por criticidade (turno completo):\n")
print(f"{'#':<3} {'Sensor':<16} {'Sala':<10} {'Média':>7} {'Max':>7} {'Alertas':>8} {'Score':>8}")
print('-' * 68)
for posicao, r in enumerate(iot.aggregate(pipeline), start=1):
    print(f"{posicao:<3} {r['sensor']:<16} {r['sala']:<10} "
          f"{r['temp_media']:>6.2f}°C "
          f"{r['temp_max']:>6.2f}°C "
          f"{r['qtd_alertas']:>8} "
          f"{r['score_criticidade']:>8.2f}")

### 5.4 Detectar períodos de aquecimento contínuo com `$setWindowFields`

Identifica janelas de 10 minutos onde a temperatura média foi a mais alta, particionando por sensor.

In [ ]:
pipeline = [
    {'$match':  {'meta.sensor_id': 'rack-B-top'}},
    {'$sort':   {'ts': 1}},
    {
        '$setWindowFields': {
            'partitionBy': '$meta.sensor_id',
            'sortBy':      {'ts': 1},
            'output': {
                'media_janela_10min': {
                    '$avg': '$temperatura_c',
                    'window': {
                        'range': [-600, 0],     # 600 segundos = 10 minutos antes do ponto atual
                        'unit':  'second'
                    }
                }
            }
        }
    },
    {'$sort':   {'media_janela_10min': -1}},
    {'$limit':  5},
    {'$project': {'_id': 0, 'ts': 1, 'temperatura_c': 1, 'media_janela_10min': 1}}
]

print("Top 5 momentos de maior calor acumulado (10 min) — rack-B-top:\n")
print(f"{'Timestamp':<22}  {'Temp Atual':>12}  {'Média 10min':>12}")
print('-' * 52)
for doc in iot.aggregate(pipeline):
    print(f"{doc['ts'].strftime('%H:%M:%S'):<22}  "
          f"{doc['temperatura_c']:>11.2f}°C  "
          f"{doc['media_janela_10min']:>11.2f}°C")

---
## Seção 6 — Índices e Performance

Time Series Collections criam automaticamente um índice composto em `(metaField, timeField)`. Entender como os índices funcionam é essencial para escalar aplicações.

### 6.1 Inspecionar índices existentes

In [ ]:
import json

indices = list(iot.list_indexes())
print(f'Índices na coleção "sensores_iot" ({len(indices)} total):\n')
for idx in indices:
    print(f"  Nome: {idx['name']}")
    print(f"  Chave: {dict(idx['key'])}")
    print()

### 6.2 Criar índice secundário para acelerar queries por localização

In [ ]:
from pymongo import ASCENDING

# Índice no campo de metadados para filtros por sala
iot.create_index([('meta.sala', ASCENDING)], name='idx_sala')

print('Índice idx_sala criado.')

indices_atualizados = list(iot.list_indexes())
print(f'Total de índices agora: {len(indices_atualizados)}')
for idx in indices_atualizados:
    print(f"  {idx['name']}: {dict(idx['key'])}")

### 6.3 Analisar o plano de execução de uma query com `explain`

`explain` revela se a query usa um índice (`IXSCAN`) ou faz varredura completa (`COLLSCAN`). Sempre prefira `IXSCAN` em coleções grandes.

In [ ]:
explicacao = db.command(
    'explain',
    {
        'find': 'sensores_iot',
        'filter': {
            'meta.sala': 'sala-1',
            'ts': {
                '$gte': datetime(2024, 1, 15, 9, 0, tzinfo=timezone.utc),
                '$lt':  datetime(2024, 1, 15, 10, 0, tzinfo=timezone.utc)
            }
        }
    },
    verbosity='queryPlanner'
)

plano = explicacao.get('queryPlanner', {}).get('winningPlan', {})

def extrair_stages(plano, profundidade=0):
    stage = plano.get('stage', 'N/A')
    print('  ' * profundidade + f'Stage: {stage}')
    if 'indexName' in plano:
        print('  ' * profundidade + f'  Índice usado: {plano["indexName"]}')
    if 'inputStage' in plano:
        extrair_stages(plano['inputStage'], profundidade + 1)

print('Plano de execução:')
extrair_stages(plano)

### 6.4 TTL — Expiração automática de dados antigos

Time Series Collections suportam expiração automática via `expireAfterSeconds`. Dados mais antigos que o limite são removidos automaticamente pelo MongoDB em background — ideal para manter apenas uma janela de retenção sem jobs de limpeza manuais.

In [ ]:
# Exemplo: coleção que retém dados por 90 dias
RETENCAO_DIAS = 90

if 'sensores_retencao' in db.list_collection_names():
    db.drop_collection('sensores_retencao')

db.create_collection(
    'sensores_retencao',
    timeseries={
        'timeField':   'ts',
        'metaField':   'meta',
        'granularity': 'hours'
    },
    expireAfterSeconds=RETENCAO_DIAS * 24 * 3600
)

print(f'Coleção criada com TTL de {RETENCAO_DIAS} dias.')
print('O MongoDB removerá automaticamente documentos com ts mais antigo que esse limite.')

---
## Resumo

| Conceito | Operador / Método | Quando usar |
|---|---|---|
| Criar coleção TS | `create_collection(..., timeseries={...})` | Sempre que o campo principal for um timestamp |
| Filtro por tempo | `$gte` / `$lte` no `timeField` | Base de qualquer query TS |
| Downsampling | `$dateTrunc` + `$group` | Reduzir resolução para dashboards ou arquivamento |
| Média móvel | `$setWindowFields` + `$avg` com `documents: [-N, 0]` | Suavizar ruídos em séries |
| Soma acumulada | `$setWindowFields` + `$sum` com `unbounded` | Totais progressivos |
| Delta entre pontos | `$shift` dentro de `$setWindowFields` | Detectar variações bruscas |
| Percentis | `$percentile` dentro de `$group` | Definir limiares baseados em dados reais |
| TTL automático | `expireAfterSeconds` na criação | Política de retenção sem cron jobs |
| Performance | `explain()` | Validar uso de índice antes de ir para produção |

### Próximos passos

- Conectar esses dados ao **Grafana** (notebook `3.grafana.ipynb`) para visualização em tempo real
- Usar **Telegraf** (notebook `2.telegraf.ipynb`) para ingerir métricas reais do sistema nessas coleções
- Explorar a [documentação oficial de Time Series Collections](https://www.mongodb.com/docs/manual/core/timeseries-collections/)

In [ ]:
# Encerrar a conexão com o MongoDB
client.close()
print('Conexão encerrada.')